In [0]:
%run "../../utils/00_config"

In [0]:
from pyspark.sql import functions as F

# Gold intermediate source
price_quality = spark.table(
    get_gold_table("intermediate", "int_item_price_quality")
)

# Silver item dimension
dim_item = spark.table(
    get_table("intermediate", "dim_item")
)

print(f"Price quality records: {price_quality.count():,}")
print(f"Item dimension records: {dim_item.count():,}")

# display(dim_item)

In [0]:
# Select item attributes
item_attributes = dim_item.select(
    "item_id",
    "item_name",
    "category",
    "is_set_root",
    "set_part_ids",
    "ducats",
    "required_mastery_rank"
)

# Enrich price observations
mart_item_price_history = (
    price_quality
    .join(
        item_attributes,
        on="item_id",
        how="left"
    )
)

print(f"Source records: {price_quality.count():,}")
print(f"Mart records: {mart_item_price_history.count():,}")

# display(mart_item_price_history.limit(10))

In [0]:
# Validate dimension uniqueness
duplicate_items = (
    dim_item
    .groupBy("item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_items == 0, (
    f"Found {duplicate_items} duplicated item_id values in dim_item."
)

# Validate source key uniqueness
duplicate_orders = (
    price_quality
    .groupBy("execution_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_orders == 0, (
    f"Found {duplicate_orders} duplicated source keys."
)

# Identify offers without item dimension match
missing_items = (
    price_quality
    .join(
        dim_item.select("item_id"),
        on="item_id",
        how="left_anti"
    )
    .count()
)

# Validate enrichment
validation = mart_item_price_history.agg(
    F.count("*").alias("total_records"),
    F.countDistinct(
        F.struct("execution_id", "order_id")
    ).alias("unique_keys"),
    F.sum(
        F.when(F.col("item_name").isNull(), 1).otherwise(0)
    ).alias("missing_item_names"),
    F.sum(
        F.when(F.col("category").isNull(), 1).otherwise(0)
    ).alias("missing_categories"),
    F.sum(
        F.when(F.col("is_set_root").isNull(), 1).otherwise(0)
    ).alias("missing_set_flags")
)

display(validation)

print(f"Duplicated dimension keys: {duplicate_items}")
print(f"Duplicated source keys: {duplicate_orders}")
print(f"Offers without item dimension match: {missing_items}")

assert duplicate_items == 0
assert duplicate_orders == 0
assert mart_item_price_history.count() == price_quality.count()
assert missing_items == 0

print("Mart enrichment validation passed.")


In [0]:
# Build item price history mart
mart_item_price_history = (
    mart_item_price_history
    .select(
        # Observation identifiers
        "execution_id",
        "order_id",
        "item_id",

        # Item attributes
        "item_slug",
        "item_name",
        "product_slug",
        "category",
        "is_set_root",

        # Observation timestamps
        "observed_at",
        "order_created_at",
        "order_updated_at",

        # Offer attributes
        "order_type",
        "platinum",
        "quantity",
        "per_trade",

        # Player attributes
        "player_id",
        "player_name",
        "player_status",
        "platform",
        "crossplay",
        "locale",
        "player_reputation",

        # Price quality
        "price_quality_status",
        "is_price_outlier",
        "total_orders",
        "q1",
        "q3",
        "iqr",
        "lower_bound",
        "upper_bound",
        "quality_status",

        # Item metadata
        "set_part_ids",
        "ducats",
        "required_mastery_rank"
    )
    .withColumn(
        "processing_timestamp",
        F.current_timestamp()
    )
)

# display(mart_item_price_history.limit(10))

In [0]:
from delta.tables import DeltaTable

MART_TARGET = get_gold_table(
    "marts",
    "mart_item_price_history"
)

merge_condition = """
    target.execution_id = source.execution_id
    AND target.order_id = source.order_id
"""

if spark.catalog.tableExists(MART_TARGET):
    (
        DeltaTable.forName(spark, MART_TARGET)
        .alias("target")
        .merge(
            mart_item_price_history.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        mart_item_price_history.write
        .format("delta")
        .mode("append")
        .saveAsTable(MART_TARGET)
    )

print(f"Table successfully materialized: {MART_TARGET}")


In [0]:

mart_result = spark.table(MART_TARGET)

expected_count = mart_item_price_history.count()
actual_count = mart_result.count()

duplicate_count = (
    mart_result
    .groupBy("execution_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert expected_count == actual_count, (
    f"Materialization mismatch: expected={expected_count}, "
    f"actual={actual_count}"
)

assert duplicate_count == 0, (
    f"Found {duplicate_count} duplicate keys."
)

print(f"Expected records: {expected_count:,}")
print(f"Materialized records: {actual_count:,}")
print(f"Duplicate keys: {duplicate_count}")
print("Gold mart materialization validation passed.")

display(
    mart_result
    .groupBy("price_quality_status")
    .count()
    .orderBy("price_quality_status")
)
